In [0]:
%sql
CREATE CATALOG IF NOT EXISTS CleanStream;

CREATE SCHEMA IF NOT EXISTS CleanStream.BRONZE;
CREATE SCHEMA IF NOT EXISTS CleanStream.SILVER;
CREATE SCHEMA IF NOT EXISTS CleanStream.GOLD;


CREATE VOLUME IF NOT EXISTS CleanStream.BRONZE.KAFKA_CHECKPOINT;

# BRONZE 

### GOAL 
- Connect DataBricks with Kafka ( Auth )
- Load Data
- Add ingestion time
- Save Raw Data

In [0]:
# Auth Info 
BOOTSTRAP_SERVER = "************"
API_KEY = "************"
API_SECRET = "************"
TOPIC = "event_data"
CHECKPOINT_PATH = "/Volumes/CleanStream/BRONZE/KAFKA_CHECKPOINT/RAW_DATA_STREAM"


In [0]:
from pyspark.sql.functions import col, current_timestamp

# Step 1 : Stop the running Kafka Streams 
for q in spark.streams.active:
    q.stop()


# Step 2 : Load Stream From kafka
bronze_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option(
        "kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{API_KEY}" password="{API_SECRET}";'
    )
    .load()
    .select(
        col("key").cast("string").alias("kafka_key"),
        col("value").cast("string").alias("raw_json"),
        col("topic").alias("kafka_topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .withColumn("ingestion_time", current_timestamp())
)

# Step 3 : Write the Data
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable("CleanStream.BRONZE.RAW_DATA")
)

# Step 4: Call the stream until it is terminated
query.awaitTermination()

In [0]:
display(spark.sql("""
SELECT *
FROM CleanStream.BRONZE.RAW_DATA
ORDER BY kafka_timestamp DESC
LIMIT 50
"""))

kafka_key,raw_json,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_time
evt_10079,"{""event_id"": ""evt_10079"", ""customer_id"": 164, ""amount"": 1566.21, ""payment_status"": ""success"", ""event_time"": ""2026-05-03T07:20:46.032682Z""}",event_data,5,242,2026-05-03T07:11:38.518Z,2026-05-03T07:13:00.814Z
evt_10078,"{""event_id"": ""evt_10078"", ""customer_id"": 671, ""amount"": 419.66, ""payment_status"": ""success"", ""event_time"": ""2026-05-03T07:20:42.032682Z""}",event_data,5,241,2026-05-03T07:11:38.364Z,2026-05-03T07:13:00.814Z
evt_10077,"{""event_id"": ""evt_10077"", ""customer_id"": 960, ""amount"": -464.49, ""payment_status"": ""pending"", ""event_time"": ""2026-05-03T07:20:30.032682Z""}",event_data,4,161,2026-05-03T07:11:38.215Z,2026-05-03T07:13:00.814Z
evt_10076,"{""event_id"": ""evt_10076"", ""customer_id"": 928, ""amount"": 894.0, ""payment_status"": ""failed"", ""event_time"": ""2026-05-03T07:20:26.032682Z""}",event_data,2,300,2026-05-03T07:11:38.064Z,2026-05-03T07:13:00.814Z
missing_key,"{""customer_id"": 205, ""amount"": 1087.03, ""payment_status"": ""success"", ""event_time"": ""2026-05-03T07:20:18.032682Z""}",event_data,2,299,2026-05-03T07:11:37.916Z,2026-05-03T07:13:00.814Z
evt_10074,"{""event_id"": ""evt_10074"", ""customer_id"": 338, ""amount"": 815.59, ""payment_status"": ""success"", ""event_time"": ""2026-05-03T07:20:11.032682Z""}",event_data,0,175,2026-05-03T07:11:37.766Z,2026-05-03T07:13:00.814Z
missing_key,"{""customer_id"": 623, ""amount"": 881.98, ""payment_status"": ""success"", ""event_time"": ""2026-05-03T07:20:01.032682Z""}",event_data,2,298,2026-05-03T07:11:37.607Z,2026-05-03T07:13:00.814Z
evt_10072,"{""event_id"": ""evt_10072"", ""customer_id"": 708, ""amount"": 1744.68, ""payment_status"": ""done"", ""event_time"": ""2026-05-03T07:19:55.032682Z""}",event_data,1,184,2026-05-03T07:11:37.455Z,2026-05-03T07:13:00.814Z
evt_10071,"{""event_id"": ""evt_10071"", ""customer_id"": 903, ""amount"": 901.37, ""payment_status"": ""pending"", ""event_time"": ""2026-05-03T07:19:53.032682Z""}",event_data,3,223,2026-05-03T07:11:37.290Z,2026-05-03T07:13:00.814Z
evt_10070,"{""event_id"": ""evt_10070"", ""customer_id"": 851, ""amount"": 662.88, ""payment_status"": ""pending"", ""event_time"": ""2026-05-03T07:19:46.032682Z""}",event_data,5,240,2026-05-03T07:11:37.138Z,2026-05-03T07:13:00.814Z


# SILVER

### 1. Required field errors
### 
WHEN REQUIRED COLUMNS ARE NOT THERE OR THE VALUE IS NULL

In [0]:
from pyspark.sql.functions import col, from_json, trim, when, lit, concat_ws, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

BRONZE_TABLE = "cleanstream.bronze.raw_data"
QUARANTINE_TABLE = "cleanstream.silver.quarantine_table"
SILVER_CLEAN_TABLE = "CleanStream.SILVER.clean_parsed_data"
SILVER_VALID_DATA_TABLE = "CleanStream.SILVER.valid_data"

schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("event_time", StringType(), True)
])


In [0]:
from pyspark.sql.functions import col


bronze_df = spark.table(BRONZE_TABLE)

if spark.catalog.tableExists(SILVER_CLEAN_TABLE):
    existing_df = (
        spark.table(SILVER_CLEAN_TABLE)
        .select("kafka_topic", "kafka_partition", "kafka_offset")
        .dropDuplicates()
    )

    new_bronze_df = (
        bronze_df.alias("b")
        .join(
            existing_df.alias("s"),
            (col("b.kafka_topic") == col("s.kafka_topic")) &
            (col("b.kafka_partition") == col("s.kafka_partition")) &
            (col("b.kafka_offset") == col("s.kafka_offset")),
            "left_anti"
        )
    )
else:
    new_bronze_df = bronze_df

new_row_count = new_bronze_df.count()

print(f"New bronze rows to process: {new_row_count}")

New bronze rows to process: 79


# STEP 1 : CONVERT JSON DATA TO COLUMN FORMAT

In [0]:
from pyspark.sql.functions import col, from_json, map_keys, array_except, array, lit, size,lower
from pyspark.sql.types import MapType, StringType

def add_schema_drift_columns(df):
    required_fields = ["event_id", "customer_id", "amount", "payment_status", "event_time"]
    json_map_schema = MapType(StringType(), StringType())

    return (
        df
        .withColumn("json_map_for_drift", from_json(col("raw_json"), json_map_schema))
        .withColumn(
            "extra_fields",
            array_except(
                map_keys(col("json_map_for_drift")),
                array(*[lit(x) for x in required_fields])
            )
        )
        .withColumn(
            "schema_drift_exists",
            size(col("extra_fields")) > 0
        )
        .drop("json_map_for_drift")
    )

In [0]:
bronze_df = new_row_count

# Convert that into datetime column
parsed_df = new_bronze_df.withColumn("data", from_json(col("raw_json"),schema))

# Add schema drift columns - when we have schema drift, we will have extra columns in the json
parsed_df = add_schema_drift_columns(parsed_df)

parsed_df = parsed_df.select(
    "kafka_key",
    "raw_json",
    "kafka_topic",
    "kafka_partition",
    "kafka_offset",
    "kafka_timestamp",
    "ingestion_time",
    "extra_fields",
    "schema_drift_exists",
    trim(col("data.event_id")).alias("parsed_event_id"),
    trim(col("data.customer_id")).alias("parsed_customer_id"),
    trim(col("data.amount")).alias("parsed_amount"),
    lower(trim(col("data.payment_status"))).alias("parsed_payment_status"),
    trim(col("data.event_time")).alias("parsed_event_time")
)


parsed_df.show(20, truncate=False)

+-----------+-------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+------------+-----------------------+-----------------------+------------+-------------------+---------------+------------------+-------------+---------------------+---------------------------+
|kafka_key  |raw_json                                                                                                                                   |kafka_topic|kafka_partition|kafka_offset|kafka_timestamp        |ingestion_time         |extra_fields|schema_drift_exists|parsed_event_id|parsed_customer_id|parsed_amount|parsed_payment_status|parsed_event_time          |
+-----------+-------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+------------+-----------------------+-----------------------+--------

# STEP - 2 : FILTER NEW ROWS 

# REQUIRED COLUMN CHECK

In [0]:
from pyspark.sql.functions import col, trim, lower, regexp_replace, when, coalesce, expr, from_unixtime, length

def normalize_event_time(df, input_col="parsed_event_time", output_col="clean_event_time"):
    df = (
        df

        # Step 1: clean raw string
        .withColumn("event_time_text", trim(col(input_col).cast("string")))

        # Step 2: remove junk values
        .withColumn(
            "event_time_text",
            when(
                (col("event_time_text").isNull()) |
                (col("event_time_text") == "") |
                (lower(col("event_time_text")).isin("null", "none", "nan", "invalid-date")),
                None
            ).otherwise(col("event_time_text"))
        )

        # Step 3: fix ISO timestamps (THIS IS THE KEY FIX)
        # 2026-04-24T21:01:40.999658Z → 2026-04-24 21:01:40.999658
        .withColumn(
            "event_time_fixed",
            regexp_replace(
                regexp_replace(col("event_time_text"), "T", " "),
                "Z",
                ""
            )
        )

        # Step 4: parse everything safely
        .withColumn(
            output_col,
            coalesce(
                expr("try_cast(event_time_fixed as timestamp)"),

                # unix seconds
                when(
                    length(col("event_time_text")) == 10,
                    from_unixtime(col("event_time_text").cast("long")).cast("timestamp")
                ),

                # unix milliseconds
                when(
                    length(col("event_time_text")) == 13,
                    from_unixtime((col("event_time_text").cast("long") / 1000)).cast("timestamp")
                )
            )
        )
    )

    return df


In [0]:

parsed_df = normalize_event_time(parsed_df, input_col="parsed_event_time", output_col="clean_event_time")


required_column_checked = parsed_df.withColumn(
  "schema_error_summary",
  concat_ws(
    " , ",
    when((col("parsed_event_id").isNull()) | (col("parsed_event_id") == ''), lit("parsed_event_id is missing")),
    when((col("parsed_customer_id").isNull()) | (col("parsed_customer_id") == ''), lit("parsed_customer_id is missing")),
    when((col("parsed_amount").isNull()) | (col("parsed_amount") == ''), lit("parsed_amount is missing")),
    when((col("parsed_payment_status").isNull()) | (col("parsed_payment_status") == ''), lit("parsed_payment_status is missing")),
    when((col("clean_event_time").isNull() ), lit("parsed_event_time is missing"))
  )
)

# required_column_checked = required_column_checked.filter(col("schema_error_summary") == "")

required_column_checked.show(5)


+---------+--------------------+-----------+---------------+------------+--------------------+--------------------+------------+-------------------+---------------+------------------+-------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|kafka_key|            raw_json|kafka_topic|kafka_partition|kafka_offset|     kafka_timestamp|      ingestion_time|extra_fields|schema_drift_exists|parsed_event_id|parsed_customer_id|parsed_amount|parsed_payment_status|   parsed_event_time|     event_time_text|    event_time_fixed|    clean_event_time|schema_error_summary|
+---------+--------------------+-----------+---------------+------------+--------------------+--------------------+------------+-------------------+---------------+------------------+-------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|evt_10001|{"event_id": "

#STEP - 3 : FIX DATA TYPES

In [0]:
from pyspark.sql.functions import col, regexp_replace, regexp_extract, when, lit, concat_ws, trim


# FIX THE AMOUNT COLUMN
required_column_checked = required_column_checked.withColumn(
    "clean_amount", regexp_replace(trim(col("parsed_amount")), "[^0-9.-]", "")).withColumn(
        "clean_parsed_amount", when( col('clean_amount') == "", None).otherwise(
            col('clean_amount').cast("double")
        )
    )

required_column_checked = required_column_checked.withColumn(
    "clean_amount_issue_exists", when(
        (col("clean_amount") != col("parsed_amount")), lit("true")
        ).otherwise(lit("false")))

required_column_checked = required_column_checked.withColumn(
    "clean_amount_issue_fixed", when(
        (col("clean_amount") != col("parsed_amount")) & (col("clean_parsed_amount").isNotNull()) , lit("true")
        ).otherwise(lit("false")))


# FIX THE CUSTOMER ID COLUMN
required_column_checked = required_column_checked.withColumn(
    "clean_customer_id", regexp_replace(trim(col("parsed_customer_id")), "[^0-9.-]", "")).withColumn(
        "clean_parsed_customer_id", when( (col('clean_customer_id') == "")| (col('clean_customer_id') == "guest"), None ).otherwise(
            col('clean_customer_id').cast("int")
        )
    )

required_column_checked = required_column_checked.withColumn(
    "clean_customer_id_issue_exists", when(
        (col("clean_customer_id") != col("parsed_customer_id")), lit("true")
        ).otherwise(lit("false")))


required_column_checked = required_column_checked.withColumn(
    "clean_customer_id_issue_fixed", when(
        (col("clean_customer_id") != col("parsed_customer_id")) & (col("clean_parsed_customer_id").isNotNull()) , lit("true")
        ).otherwise(lit("false")))

# add the error cause - data which is not still fixed yet

clean_rows_checked = required_column_checked.withColumn(
  "clean_error_summary",
  concat_ws(
    " , ",
    when((col("clean_parsed_amount").isNull())  , lit("event_id is not clean")),
    when((col("clean_parsed_customer_id").isNull()), lit("customer_id is not clean"))
  )
)

clean_rows_checked.show(5)

+---------+--------------------+-----------+---------------+------------+--------------------+--------------------+------------+-------------------+---------------+------------------+-------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+-------------------+-------------------------+------------------------+-----------------+------------------------+------------------------------+-----------------------------+--------------------+
|kafka_key|            raw_json|kafka_topic|kafka_partition|kafka_offset|     kafka_timestamp|      ingestion_time|extra_fields|schema_drift_exists|parsed_event_id|parsed_customer_id|parsed_amount|parsed_payment_status|   parsed_event_time|     event_time_text|    event_time_fixed|    clean_event_time|schema_error_summary|clean_amount|clean_parsed_amount|clean_amount_issue_exists|clean_amount_issue_fixed|clean_customer_id|clean_parsed_customer_id|clean_customer_

In [0]:
clean_rows_checked.select("clean_error_summary").filter(col("clean_error_summary") != '').show()

+--------------------+
| clean_error_summary|
+--------------------+
|customer_id is no...|
|customer_id is no...|
|event_id is not c...|
|customer_id is no...|
|event_id is not c...|
|event_id is not c...|
+--------------------+



In [0]:
clean_rows_checked.count()

79

# STEP - 4 :  Business rule errors


In [0]:

from pyspark.sql.functions import col, current_timestamp


business_validation = clean_rows_checked.withColumn(
  "business_validation_errors",
  concat_ws(
    " , ",
    when((col("clean_parsed_amount") < 0), lit("amount is negative")), # ignore the row where the amount is negative
    when((col('clean_event_time') > current_timestamp()), lit("event_time is in the future")), # when the event time is worng
    when( (~col("parsed_payment_status").isin("success","failed","pending") ), lit("payment_status is not valid") )
  )
)

In [0]:
business_validation.count()


79

# FINAL ERROR VALIDATION

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, when, trim, concat_ws, month, year

def empty_to_null(column_name):
    return when(trim(col(column_name)) == "", None).otherwise(trim(col(column_name)))

final_data = (
    business_validation
    .withColumn(
        "final_error_summary",
        concat_ws(
            " , ",
            empty_to_null("schema_error_summary"),
            empty_to_null("clean_error_summary"),
            empty_to_null("business_validation_errors")
        )
    )
    .withColumn("Month", month(col("clean_event_time")))
    .withColumn("Year", year(col("clean_event_time")))
)

print("Rows ready to save:", final_data.count())
final_data.show(20, truncate=False)

Rows ready to save: 79
+-----------+-------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------------+------------+-----------------------+-----------------------+------------+-------------------+---------------+------------------+-------------+---------------------+---------------------------+---------------------------+--------------------------+--------------------------+--------------------------------+------------+-------------------+-------------------------+------------------------+-----------------+------------------------+------------------------------+-----------------------------+------------------------+---------------------------+--------------------------------------------------------------------------------------+-----+----+
|kafka_key  |raw_json                                                                                                                               

In [0]:
print(final_data.count())   # will still be 77

79


# CREATE TABLES FOR SILVER

In [0]:


valid_data = final_data.filter(col("final_error_summary") == "")
valid_row_count = valid_data.count()

valid_output = valid_data.select(
    col("parsed_event_id").alias("event_id"),
    col("clean_parsed_customer_id").alias("customer_id"),
    col("clean_parsed_amount").alias("amount"),
    col("parsed_payment_status").alias("payment_status"),
    col("clean_event_time").alias("event_time"),

    "kafka_topic",
    "kafka_partition",
    "kafka_offset",
    "kafka_timestamp",
    "ingestion_time",

    "schema_drift_exists",
    "extra_fields"
).withColumn("silver_processed_time", current_timestamp())


# SAVE VALID DATA
valid_output.write.format("delta").mode("append").saveAsTable(SILVER_VALID_DATA_TABLE)



# SAVE INVALID DATA
invalid_data = final_data.filter(col("final_error_summary") != "").withColumn("silver_processed_time", current_timestamp())
invalid_row_count = invalid_data.count()
invalid_data.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)

print(f"Valid rows: {valid_row_count}")
print(f"Invalid rows: {invalid_row_count}")


# SAVED THE FINAL FULL CLEAN DATA TO A TABLE
if final_data.count() > 0:
    final_data = final_data.write.format("delta").mode("append").saveAsTable(SILVER_CLEAN_TABLE)
    print(f"Total rows saved: {valid_row_count + invalid_row_count}")
else:
    print("No new rows to save. Produce new Kafka messages or reset checkpoints/table.")


Valid rows: 18
Invalid rows: 59
Total rows saved: 77


### GOLD LAYER

In [0]:
# GOLD TABLE NAMES
GOLD_SUMMARY_TABLE = "cleanstream.gold.data_quality_summary"
GOLD_ERROR_TABLE = "cleanstream.gold.error_breakdown"


In [0]:
# GOLD 1 - DATA QUALITY SUMMARY

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SUMMARY_TABLE} AS

SELECT
    COUNT(*) AS total_events,

    SUM(CASE WHEN final_error_summary IS NULL OR TRIM(final_error_summary) = '' THEN 1 ELSE 0 END) AS valid_events,

    SUM(CASE WHEN final_error_summary IS NOT NULL AND TRIM(final_error_summary) != '' THEN 1 ELSE 0 END) AS invalid_events,

    ROUND(
        SUM(CASE WHEN final_error_summary IS NULL OR TRIM(final_error_summary) = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS valid_rate,

    ROUND(
        SUM(CASE WHEN final_error_summary IS NOT NULL AND TRIM(final_error_summary) != '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS invalid_rate,

    SUM(CASE WHEN schema_drift_exists = true THEN 1 ELSE 0 END) AS schema_drift_count,

    ROUND(
        SUM(CASE WHEN schema_drift_exists = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS schema_drift_rate,

    SUM(CASE WHEN parsed_event_id IS NULL OR TRIM(parsed_event_id) = '' THEN 1 ELSE 0 END) AS missing_event_id_count,

    SUM(CASE WHEN parsed_customer_id IS NULL OR TRIM(CAST(parsed_customer_id AS STRING)) = '' THEN 1 ELSE 0 END) AS missing_customer_id_count,

    SUM(CASE WHEN parsed_amount IS NULL OR TRIM(CAST(parsed_amount AS STRING)) = '' THEN 1 ELSE 0 END) AS missing_amount_count,

    SUM(CASE WHEN parsed_payment_status IS NULL OR TRIM(parsed_payment_status) = '' THEN 1 ELSE 0 END) AS missing_payment_status_count,

    SUM(
        CASE 
            WHEN parsed_amount IS NOT NULL 
             AND TRIM(CAST(parsed_amount AS STRING)) != ''
             AND clean_parsed_amount IS NULL
            THEN 1 ELSE 0 
        END
    ) AS invalid_amount_count,

    SUM(
        CASE 
            WHEN parsed_payment_status IS NOT NULL
             AND TRIM(parsed_payment_status) != ''
             AND LOWER(parsed_payment_status) NOT IN ('success', 'failed', 'pending')
            THEN 1 ELSE 0 
        END
    ) AS invalid_payment_status_count,

    SUM(CASE WHEN business_validation_errors LIKE '%amount is negative%' THEN 1 ELSE 0 END) AS negative_amount_count,

    SUM(CASE WHEN business_validation_errors LIKE '%event_time is in the future%' THEN 1 ELSE 0 END) AS future_event_time_count,

    COUNT(DISTINCT kafka_partition) AS kafka_partition_count,

    MIN(kafka_offset) AS min_kafka_offset,
    MAX(kafka_offset) AS max_kafka_offset,

    CURRENT_TIMESTAMP() AS gold_processed_time

FROM {SILVER_CLEAN_TABLE}
""")

spark.table(GOLD_SUMMARY_TABLE).show(truncate=False)

+------------+------------+--------------+----------+------------+------------------+-----------------+----------------------+-------------------------+--------------------+----------------------------+--------------------+----------------------------+---------------------+-----------------------+---------------------+----------------+----------------+--------------------------+
|total_events|valid_events|invalid_events|valid_rate|invalid_rate|schema_drift_count|schema_drift_rate|missing_event_id_count|missing_customer_id_count|missing_amount_count|missing_payment_status_count|invalid_amount_count|invalid_payment_status_count|negative_amount_count|future_event_time_count|kafka_partition_count|min_kafka_offset|max_kafka_offset|gold_processed_time       |
+------------+------------+--------------+----------+------------+------------------+-----------------+----------------------+-------------------------+--------------------+----------------------------+--------------------+-------------

In [0]:
# GOLD 2 - ERROR BREAKDOWN

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_ERROR_TABLE} AS

WITH all_errors AS (

    SELECT
        kafka_partition,
        kafka_offset,
        raw_json,
        TRIM(single_error) AS error_type

    FROM {SILVER_CLEAN_TABLE}

    LATERAL VIEW EXPLODE(SPLIT(final_error_summary, ' , ')) error_table AS single_error

    WHERE final_error_summary IS NOT NULL
      AND TRIM(final_error_summary) != ''
),

error_count_data AS (

    SELECT
        error_type,
        COUNT(*) AS error_count,
        COUNT(DISTINCT kafka_partition) AS affected_partitions,
        MIN(kafka_offset) AS first_seen_offset,
        MAX(kafka_offset) AS last_seen_offset,
        FIRST(raw_json) AS sample_raw_json

    FROM all_errors

    WHERE error_type IS NOT NULL
      AND TRIM(error_type) != ''

    GROUP BY error_type
),

total_error_data AS (

    SELECT
        SUM(error_count) AS total_error_count
    FROM error_count_data
)

SELECT
    e.error_type,
    e.error_count,

    ROUND(e.error_count * 100.0 / t.total_error_count, 2) AS error_percentage,

    e.affected_partitions,
    e.first_seen_offset,
    e.last_seen_offset,
    e.sample_raw_json,

    CURRENT_TIMESTAMP() AS gold_processed_time

FROM error_count_data e
CROSS JOIN total_error_data t

ORDER BY e.error_count DESC
""")

spark.table(GOLD_ERROR_TABLE).show(truncate=False)

+--------------------------------+-----------+----------------+-------------------+-----------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------+-------------------------+
|error_type                      |error_count|error_percentage|affected_partitions|first_seen_offset|last_seen_offset|sample_raw_json                                                                                                                          |gold_processed_time      |
+--------------------------------+-----------+----------------+-------------------+-----------------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------+-------------------------+
|event_time is in the future     |465        |51.61           |6                  |3                |300             |{"customer_id": 333, "amount": 81